In [1]:
# Step 5.1：初始化Depth 1特征工程并加载applprev_1

from pathlib import Path
import os

import duckdb
import pandas as pd

# ==========================================
# 1. 路径配置
# ==========================================

PROJECT_ROOT = Path(
    r"D:\Risk_control project"
)

DATA_ROOT = Path(
    r"D:\Home_Credit_datasets"
)

TRAIN_DATA_DIR = (
    DATA_ROOT
    / "parquet_files"
    / "train"
)

DEPTH0_FEATURE_MART_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "depth0_feature_mart_dev_v1.parquet"
)

TEMP_DIR = (
    PROJECT_ROOT
    / "data"
    / "temp"
)

TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ==========================================
# 2. SQL路径转换函数
# ==========================================

def sql_text(value) -> str:
    """
    将Windows路径转换成DuckDB可使用的SQL字符串。
    """

    if isinstance(value, Path):
        value = value.as_posix()
    else:
        value = str(value).replace("\\", "/")

    value = value.replace(
        "'",
        "''"
    )

    return f"'{value}'"

# ==========================================
# 3. 定位applprev_1物理文件
# ==========================================

applprev_1_files = sorted(
    TRAIN_DATA_DIR.glob(
        "train_applprev_1_*.parquet"
    )
)


# 兼容数据集被直接放在train目录下的情况

if not applprev_1_files:

    alternative_train_dir = (
        DATA_ROOT
        / "train"
    )

    applprev_1_files = sorted(
        alternative_train_dir.glob(
            "train_applprev_1_*.parquet"
        )
    )


if not DEPTH0_FEATURE_MART_PATH.exists():
    raise FileNotFoundError(
        "找不到Depth 0 Feature Mart：\n"
        f"{DEPTH0_FEATURE_MART_PATH}"
    )


if not applprev_1_files:
    raise FileNotFoundError(
        "找不到train_applprev_1_*.parquet。\n"
        f"已检查：{TRAIN_DATA_DIR}"
    )


applprev_1_files_sql = (
    "["
    + ", ".join(
        sql_text(file_path)
        for file_path
        in applprev_1_files
    )
    + "]"
)


# ==========================================
# 4. 创建DuckDB连接
# ==========================================

connection = duckdb.connect()

duckdb_thread_count = min(
    8,
    os.cpu_count() or 4
)

connection.execute(
    f"SET threads = {duckdb_thread_count}"
)

connection.execute(
    "SET memory_limit = '24GB'"
)

connection.execute(
    f"""
    SET temp_directory =
        {sql_text(TEMP_DIR)}
    """
)

connection.execute(
    "SET preserve_insertion_order = false"
)


# ==========================================
# 5. 建立开发期案件主表
# ==========================================

connection.execute(
    f"""
    CREATE OR REPLACE VIEW
        depth0_feature_mart_dev AS

    SELECT
        *

    FROM read_parquet(
        {sql_text(DEPTH0_FEATURE_MART_PATH)}
    )
    """
)


connection.execute(
    """
    CREATE OR REPLACE VIEW
        base_dev_keys AS

    SELECT
        case_id,
        date_decision,
        WEEK_NUM

    FROM depth0_feature_mart_dev
    """
)


# ==========================================
# 6. 加载applprev_1并隔离Final OOT
# ==========================================

connection.execute(
    f"""
    CREATE OR REPLACE VIEW
        applprev_1_dev_raw AS

    SELECT
        history.*,

        base.date_decision
            AS current_date_decision,

        base.WEEK_NUM
            AS current_week_num

    FROM read_parquet(
        {applprev_1_files_sql},
        union_by_name = true
    ) AS history

    INNER JOIN base_dev_keys AS base
        ON history.case_id = base.case_id
    """
)


# ==========================================
# 7. 一次性质量检查
# ==========================================

base_check = connection.execute(
    """
    SELECT
        COUNT(*) AS row_count,

        COUNT(DISTINCT case_id)
            AS unique_case_id_count,

        MIN(WEEK_NUM)
            AS minimum_week,

        MAX(WEEK_NUM)
            AS maximum_week

    FROM base_dev_keys
    """
).df()


applprev_summary = connection.execute(
    """
    WITH history_by_case AS (
        SELECT
            case_id,
            COUNT(*) AS history_record_count

        FROM applprev_1_dev_raw

        GROUP BY case_id
    )

    SELECT
        (
            SELECT COUNT(*)
            FROM applprev_1_dev_raw
        ) AS history_row_count,

        COUNT(*) AS case_count_with_history,

        ROUND(
            100.0
            * COUNT(*)
            / (
                SELECT COUNT(*)
                FROM base_dev_keys
            ),
            4
        ) AS case_coverage_pct,

        ROUND(
            AVG(history_record_count),
            2
        ) AS average_records_per_case,

        MAX(history_record_count)
            AS maximum_records_per_case

    FROM history_by_case
    """
).df()


raw_columns = {
    row[0]
    for row in connection.execute(
        "DESCRIBE applprev_1_dev_raw"
    ).fetchall()
}


base_result = base_check.iloc[0]

required_columns = {
    "case_id",
    "num_group1",
    "current_date_decision",
    "current_week_num"
}


checks = {
    "Depth 0保持一行一个case_id": (
        int(base_result["row_count"])
        == 1_401_854
        and
        int(
            base_result[
                "unique_case_id_count"
            ]
        )
        == 1_401_854
    ),

    "Final OOT没有进入": (
        int(base_result["minimum_week"]) == 0
        and
        int(base_result["maximum_week"]) == 81
    ),

    "applprev_1必需字段存在": (
        required_columns
        .issubset(raw_columns)
    ),

    "applprev_1成功读取": (
        int(
            applprev_summary.loc[
                0,
                "history_row_count"
            ]
        )
        > 0
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 5.1检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


print(
    "Step 5.1检查："
    f"{len(checks)}/{len(checks)}通过"
)

print(
    "读取的applprev_1文件：",
    len(applprev_1_files)
)

print(
    "文件名：",
    [
        file_path.name
        for file_path
        in applprev_1_files
    ]
)

display(applprev_summary)

Step 5.1检查：4/4通过
读取的applprev_1文件： 2
文件名： ['train_applprev_1_0.parquet', 'train_applprev_1_1.parquet']


,history_row_count,case_count_with_history,case_coverage_pct,average_records_per_case,maximum_records_per_case
0,5833370,1113919,79.4604,5.24,20


In [3]:
# Step 5.2：将applprev_1聚合为一行一个case_id

# ==========================================
# 1. 读取字段名称和DuckDB数据类型
# ==========================================

applprev_schema = connection.execute(
    """
    DESCRIBE applprev_1_dev_raw
    """
).df()


excluded_columns = {
    "case_id",
    "num_group1",
    "current_date_decision",
    "current_week_num",
    "filename"
}


numeric_type_prefixes = (
    "TINYINT",
    "SMALLINT",
    "INTEGER",
    "BIGINT",
    "HUGEINT",
    "UTINYINT",
    "USMALLINT",
    "UINTEGER",
    "UBIGINT",
    "FLOAT",
    "DOUBLE",
    "REAL",
    "DECIMAL"
)


def quote_identifier(column_name: str) -> str:
    """
    安全引用DuckDB字段名。
    """

    escaped_name = column_name.replace(
        '"',
        '""'
    )

    return f'"{escaped_name}"'


numeric_columns = []
date_columns = []
categorical_columns = []
boolean_columns = []


for _, schema_row in applprev_schema.iterrows():

    column_name = schema_row["column_name"]
    column_type = str(
        schema_row["column_type"]
    ).upper()

    if column_name in excluded_columns:
        continue

    if (
        column_name.upper().endswith("D")
        or
        column_type.startswith("DATE")
        or
        column_type.startswith("TIMESTAMP")
    ):
        date_columns.append(
            column_name
        )

    elif column_type.startswith(
        numeric_type_prefixes
    ):
        numeric_columns.append(
            column_name
        )

    elif column_type.startswith(
        "BOOLEAN"
    ):
        boolean_columns.append(
            column_name
        )

    else:
        categorical_columns.append(
            column_name
        )

print("数值字段数：", len(numeric_columns))
print("日期字段数：", len(date_columns))
print("布尔字段数：", len(boolean_columns))
print("类别字段数：", len(categorical_columns))

数值字段数： 18
日期字段数： 7
布尔字段数： 2
类别字段数： 12


In [5]:
# Step 5.2.2：过滤决策时点之后才创建的记录

raw_column_names = set(
    applprev_schema["column_name"]
)


PIT_ANCHOR_COLUMN = (
    "creationdate_885D"
)


if PIT_ANCHOR_COLUMN in raw_column_names:

    pit_anchor = quote_identifier(
        PIT_ANCHOR_COLUMN
    )

    future_created_record_count = (
        connection.execute(
            f"""
            SELECT
                COUNT(*)

            FROM applprev_1_dev_raw

            WHERE
                TRY_CAST(
                    {pit_anchor}
                    AS DATE
                )
                >
                CAST(
                    current_date_decision
                    AS DATE
                )
            """
        ).fetchone()[0]
    )

    connection.execute(
        f"""
        CREATE OR REPLACE VIEW
            applprev_1_dev_pit AS

        SELECT
            *

        FROM applprev_1_dev_raw

        WHERE
            {pit_anchor} IS NULL

            OR

            TRY_CAST(
                {pit_anchor}
                AS DATE
            )
            <=
            CAST(
                current_date_decision
                AS DATE
            )
        """
    )

else:

    future_created_record_count = 0

    connection.execute(
        """
        CREATE OR REPLACE VIEW
            applprev_1_dev_pit AS

        SELECT
            *

        FROM applprev_1_dev_raw
        """
    )


print(
    "决策时点之后创建并被排除的记录数：",
    f"{future_created_record_count:,}"
)

决策时点之后创建并被排除的记录数： 125,999


In [6]:
# 修正版Step 5.2.3：低内存历史申请聚合

import gc


# ==========================================
# 1. 清理上次失败产生的临时对象
# ==========================================

connection.execute(
    """
    DROP TABLE IF EXISTS
        applprev_1_features_dev
    """
)

gc.collect()


# ==========================================
# 2. 设置适合32GB内存的DuckDB参数
# ==========================================

connection.execute(
    "SET threads = 2"
)

connection.execute(
    "SET memory_limit = '20GB'"
)

connection.execute(
    "SET preserve_insertion_order = false"
)


print(
    "DuckDB低内存聚合模式已启用。"
)


# ==========================================
# 3. 构建基础聚合表达式
# ==========================================

aggregation_expressions = [
    """
    COUNT(*) AS
        applprev1__record_count
    """
]


# ==========================================
# 4. 数值字段
# ==========================================

for column_name in numeric_columns:

    column_sql = quote_identifier(
        column_name
    )

    prefix = (
        f"applprev1__{column_name}"
    )

    aggregation_expressions.extend(
        [
            f"""
            CAST(
                AVG(
                    CAST(
                        {column_sql}
                        AS DOUBLE
                    )
                )
                AS FLOAT
            ) AS "{prefix}__mean"
            """,

            f"""
            CAST(
                MIN(
                    CAST(
                        {column_sql}
                        AS DOUBLE
                    )
                )
                AS FLOAT
            ) AS "{prefix}__min"
            """,

            f"""
            CAST(
                MAX(
                    CAST(
                        {column_sql}
                        AS DOUBLE
                    )
                )
                AS FLOAT
            ) AS "{prefix}__max"
            """
        ]
    )

    # 以A结尾的Home Credit字段通常是金额字段。
    # 金额字段额外生成历史合计值。

    if column_name.endswith("A"):

        aggregation_expressions.append(
            f"""
            CAST(
                SUM(
                    CAST(
                        {column_sql}
                        AS DOUBLE
                    )
                )
                AS FLOAT
            ) AS "{prefix}__sum"
            """
        )


# ==========================================
# 5. 日期字段
# ==========================================

for column_name in date_columns:

    column_sql = quote_identifier(
        column_name
    )

    prefix = (
        f"applprev1__{column_name}"
    )

    days_expression = f"""
        DATE_DIFF(
            'day',
            TRY_CAST(
                {column_sql}
                AS DATE
            ),
            CAST(
                current_date_decision
                AS DATE
            )
        )
    """

    aggregation_expressions.extend(
        [
            f"""
            MIN(
                {days_expression}
            ) AS "{prefix}__days_to_decision_min"
            """,

            f"""
            MAX(
                {days_expression}
            ) AS "{prefix}__days_to_decision_max"
            """,

            f"""
            CAST(
                AVG(
                    {days_expression}
                )
                AS FLOAT
            ) AS "{prefix}__days_to_decision_mean"
            """
        ]
    )


# ==========================================
# 6. 布尔字段
# ==========================================

for column_name in boolean_columns:

    column_sql = quote_identifier(
        column_name
    )

    prefix = (
        f"applprev1__{column_name}"
    )

    aggregation_expressions.append(
        f"""
        CAST(
            AVG(
                CASE
                    WHEN {column_sql} = TRUE
                    THEN 1.0

                    WHEN {column_sql} = FALSE
                    THEN 0.0

                    ELSE NULL
                END
            )
            AS FLOAT
        ) AS "{prefix}__true_rate"
        """
    )


# ==========================================
# 7. 类别字段
# ==========================================

for column_name in categorical_columns:

    column_sql = quote_identifier(
        column_name
    )

    prefix = (
        f"applprev1__{column_name}"
    )

    # 不再对每个case_id执行COUNT(DISTINCT)。
    # 仅保留num_group1最小记录对应的类别值。

    aggregation_expressions.append(
        f"""
        ARG_MIN(
            {column_sql},
            num_group1
        ) AS "{prefix}__lowest_group_index_value"
        """
    )


aggregation_sql = (
    ",\n".join(
        aggregation_expressions
    )
)


print(
    "准备生成的聚合表达式数：",
    len(aggregation_expressions)
)


# ==========================================
# 8. 执行聚合
# ==========================================

connection.execute(
    f"""
    CREATE OR REPLACE TEMPORARY TABLE
        applprev_1_features_dev AS

    SELECT
        case_id,

        {aggregation_sql}

    FROM applprev_1_dev_pit

    GROUP BY
        case_id
    """
)


print(
    "applprev_1历史特征聚合完成。"
)

DuckDB低内存聚合模式已启用。
准备生成的聚合表达式数： 101


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

applprev_1历史特征聚合完成。


In [7]:
# Step 5.2.4：检查历史申请聚合结果

aggregation_check = connection.execute(
    """
    SELECT
        COUNT(*) AS feature_row_count,

        COUNT(DISTINCT case_id)
            AS unique_case_id_count,

        SUM(applprev1__record_count)
            AS aggregated_history_row_count,

        MIN(applprev1__record_count)
            AS minimum_records_per_case,

        MAX(applprev1__record_count)
            AS maximum_records_per_case

    FROM applprev_1_features_dev
    """
).df()


filtered_history_row_count = (
    connection.execute(
        """
        SELECT COUNT(*)
        FROM applprev_1_dev_pit
        """
    ).fetchone()[0]
)


feature_column_count = len(
    connection.execute(
        """
        DESCRIBE applprev_1_features_dev
        """
    ).fetchall()
)


check_row = aggregation_check.iloc[0]


checks = {
    "聚合后一行一个case_id": (
        int(check_row["feature_row_count"])
        ==
        int(check_row["unique_case_id_count"])
    ),

    "历史记录没有在聚合中丢失": (
        int(
            check_row[
                "aggregated_history_row_count"
            ]
        )
        ==
        int(filtered_history_row_count)
    ),

    "每个案件至少有一条历史记录": (
        int(
            check_row[
                "minimum_records_per_case"
            ]
        )
        >= 1
    ),

    "成功生成历史特征": (
        feature_column_count > 2
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 5.2检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


print(
    "Step 5.2检查："
    f"{len(checks)}/{len(checks)}通过"
)

print(
    "聚合后案件数：",
    f"{int(check_row['feature_row_count']):,}"
)

print(
    "聚合特征表字段数：",
    feature_column_count
)

display(aggregation_check)

Step 5.2检查：4/4通过
聚合后案件数： 1,101,236
聚合特征表字段数： 102


,feature_row_count,unique_case_id_count,aggregated_history_row_count,minimum_records_per_case,maximum_records_per_case
0,1101236,1101236,5707371.0,1,20


In [9]:
# Step 5.3：导出applprev_1历史聚合特征

DEPTH1_FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "depth1"
)

DEPTH1_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


APPLPREV1_FEATURE_PATH = (
    DEPTH1_FEATURE_DIR
    / "applprev_1_features_dev_v1.parquet"
)

FORCE_REBUILD_APPLPREV1 = True


if (
    FORCE_REBUILD_APPLPREV1
    and
    APPLPREV1_FEATURE_PATH.exists()
):
    APPLPREV1_FEATURE_PATH.unlink()

    print(
        "旧applprev_1派生文件已删除，"
        "准备按正确日期类型重建。"
    )


if APPLPREV1_FEATURE_PATH.exists():

    print(
        "applprev_1特征文件已经存在，"
        "跳过重复导出。"
    )

else:

    connection.execute(
        f"""
        COPY (
            SELECT
                *

            FROM applprev_1_features_dev
        )

        TO {sql_text(APPLPREV1_FEATURE_PATH)}

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE 100000
        )
        """
    )

    print(
        "applprev_1历史特征导出成功。"
    )

旧applprev_1派生文件已删除，准备按正确日期类型重建。


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

applprev_1历史特征导出成功。


In [10]:
# Step 5.3导出检查

export_check = connection.execute(
    f"""
    SELECT
        COUNT(*) AS row_count,

        COUNT(DISTINCT case_id)
            AS unique_case_id_count,

        SUM(applprev1__record_count)
            AS history_record_count

    FROM read_parquet(
        {sql_text(APPLPREV1_FEATURE_PATH)}
    )
    """
).df()


export_column_count = len(
    connection.execute(
        f"""
        DESCRIBE
        SELECT *

        FROM read_parquet(
            {sql_text(APPLPREV1_FEATURE_PATH)}
        )
        """
    ).fetchall()
)


export_row = export_check.iloc[0]


export_checks = {
    "导出行数正确": (
        int(export_row["row_count"])
        == 1_101_236
    ),

    "导出后一行一个case_id": (
        int(export_row["unique_case_id_count"])
        == 1_101_236
    ),

    "历史记录汇总数正确": (
        int(export_row["history_record_count"])
        == 5_707_371
    ),

    "导出字段数正确": (
        export_column_count 
        == feature_column_count
    )
}


failed_export_checks = [
    check_name
    for check_name, passed
    in export_checks.items()
    if not passed
]


if failed_export_checks:
    raise ValueError(
        "Step 5.3导出检查失败：\n- "
        + "\n- ".join(failed_export_checks)
    )


file_size_mb = (
    APPLPREV1_FEATURE_PATH.stat().st_size
    / 1024**2
)


print(
    "Step 5.3导出检查："
    f"{len(export_checks)}/"
    f"{len(export_checks)}通过"
)

print(
    "输出文件：",
    APPLPREV1_FEATURE_PATH
)

print(
    "文件大小：",
    f"{file_size_mb:.2f} MB"
)

print(
    "数据形状：",
    (
        int(export_row["row_count"]),
        export_column_count
    )
)

Step 5.3导出检查：4/4通过
输出文件： D:\Risk_control project\data\processed\depth1\applprev_1_features_dev_v1.parquet
文件大小： 123.83 MB
数据形状： (1101236, 102)


In [11]:
# Step 5.4：将applprev_1历史特征接入Depth 0

connection.execute(
    f"""
    CREATE OR REPLACE VIEW
        feature_mart_applprev1_dev AS

    SELECT
        base.*,

        CASE
            WHEN history.case_id IS NOT NULL
            THEN 1
            ELSE 0
        END AS has_applprev_1_history,

        CAST(
            COALESCE(
                history.applprev1__record_count,
                0
            )
            AS INTEGER
        ) AS applprev1__record_count,

        history.* EXCLUDE (
            case_id,
            applprev1__record_count
        )

    FROM read_parquet(
        {sql_text(DEPTH0_FEATURE_MART_PATH)}
    ) AS base

    LEFT JOIN read_parquet(
        {sql_text(APPLPREV1_FEATURE_PATH)}
    ) AS history
        ON base.case_id = history.case_id
    """
)


print(
    "applprev_1特征连接视图创建成功。"
)

applprev_1特征连接视图创建成功。


In [12]:
# Step 5.4：正式连接后的粒度检查

join_check = connection.execute(
    """
    SELECT
        COUNT(*) AS row_count,

        COUNT(DISTINCT case_id)
            AS unique_case_id_count,

        SUM(has_applprev_1_history)
            AS case_count_with_history,

        SUM(
            CASE
                WHEN has_applprev_1_history = 0
                THEN 1
                ELSE 0
            END
        ) AS case_count_without_history,

        SUM(applprev1__record_count)
            AS total_history_record_count

    FROM feature_mart_applprev1_dev
    """
).df()


depth0_column_count = len(
    connection.execute(
        f"""
        DESCRIBE
        SELECT *

        FROM read_parquet(
            {sql_text(DEPTH0_FEATURE_MART_PATH)}
        )
        """
    ).fetchall()
)


joined_column_count = len(
    connection.execute(
        """
        DESCRIBE feature_mart_applprev1_dev
        """
    ).fetchall()
)


# 87个非主键历史字段，加1个历史存在标记

# applprev_1排除case_id后接入，
# 同时增加一个has_applprev_1_history，
# 因此净增加列数等于导出文件总列数。

expected_joined_column_count = (
    depth0_column_count
    + export_column_count
)


join_row = join_check.iloc[0]


checks = {
    "LEFT JOIN没有改变样本行数": (
        int(join_row["row_count"])
        == 1_401_854
    ),

    "连接后仍一行一个case_id": (
        int(join_row["unique_case_id_count"])
        == 1_401_854
    ),

    "有效历史案件数正确": (
        int(join_row["case_count_with_history"])
        == 1_101_236
    ),

    "历史记录总数正确": (
        int(join_row["total_history_record_count"])
        == 5_707_371
    ),

    "连接后字段数正确": (
        joined_column_count
        == expected_joined_column_count
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 5.4检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


print(
    "Step 5.4检查："
    f"{len(checks)}/{len(checks)}通过"
)

print(
    "连接后数据形状：",
    (
        int(join_row["row_count"]),
        joined_column_count
    )
)

print(
    "有有效历史申请的案件：",
    f"{int(join_row['case_count_with_history']):,}"
)

print(
    "无有效历史申请的案件：",
    f"{int(join_row['case_count_without_history']):,}"
)

display(join_check)

Step 5.4检查：5/5通过
连接后数据形状： (1401854, 327)
有有效历史申请的案件： 1,101,236
无有效历史申请的案件： 300,618


,row_count,unique_case_id_count,case_count_with_history,case_count_without_history,total_history_record_count
0,1401854,1401854,1101236.0,300618.0,5707371.0


In [13]:
# Step 5.5：加载applprev_2并连接有效父记录

# ==========================================
# 1. 定位applprev_2文件
# ==========================================

applprev_2_files = sorted(
    TRAIN_DATA_DIR.glob(
        "train_applprev_2*.parquet"
    )
)


if not applprev_2_files:

    alternative_train_dir = (
        DATA_ROOT
        / "train"
    )

    applprev_2_files = sorted(
        alternative_train_dir.glob(
            "train_applprev_2*.parquet"
        )
    )


if not applprev_2_files:
    raise FileNotFoundError(
        "找不到train_applprev_2*.parquet"
    )


applprev_2_files_sql = (
    "["
    + ", ".join(
        sql_text(file_path)
        for file_path
        in applprev_2_files
    )
    + "]"
)


# ==========================================
# 2. 建立有效父申请键
# ==========================================

connection.execute(
    """
    CREATE OR REPLACE VIEW
        valid_applprev_1_parent_keys AS

    SELECT DISTINCT
        case_id,
        num_group1

    FROM applprev_1_dev_pit
    """
)


# ==========================================
# 3. 连接Depth 2子记录
# ==========================================

connection.execute(
    f"""
    CREATE OR REPLACE VIEW
        applprev_2_dev_raw AS

    SELECT
        child.*

    FROM read_parquet(
        {applprev_2_files_sql},
        union_by_name = true
    ) AS child

    INNER JOIN
        valid_applprev_1_parent_keys
        AS parent

        ON child.case_id
            = parent.case_id

        AND child.num_group1
            = parent.num_group1
    """
)


print(
    "applprev_2有效子记录视图创建成功。"
)

applprev_2有效子记录视图创建成功。


In [14]:
# Step 5.5：字段与粒度摘要

applprev_2_schema = connection.execute(
    """
    DESCRIBE applprev_2_dev_raw
    """
).df()


required_columns = {
    "case_id",
    "num_group1",
    "num_group2"
}


actual_columns = set(
    applprev_2_schema["column_name"]
)


if not required_columns.issubset(
    actual_columns
):
    raise ValueError(
        "applprev_2缺少必要层级字段："
        + str(
            sorted(
                required_columns
                - actual_columns
            )
        )
    )


applprev_2_feature_schema = (
    applprev_2_schema.loc[
        ~applprev_2_schema[
            "column_name"
        ].isin(
            {
                "case_id",
                "num_group1",
                "num_group2",
                "filename"
            }
        ),
        [
            "column_name",
            "column_type"
        ]
    ]
    .reset_index(drop=True)
)


applprev_2_summary = connection.execute(
    """
    WITH child_count_by_parent AS (
        SELECT
            case_id,
            num_group1,

            COUNT(*) AS
                child_record_count

        FROM applprev_2_dev_raw

        GROUP BY
            case_id,
            num_group1
    )

    SELECT
        SUM(child_record_count)
            AS child_row_count,

        COUNT(*)
            AS parent_application_count,

        COUNT(DISTINCT case_id)
            AS case_count_with_child_history,

        ROUND(
            AVG(child_record_count),
            2
        ) AS average_children_per_parent,

        MAX(child_record_count)
            AS maximum_children_per_parent

    FROM child_count_by_parent
    """
).df()


summary_row = applprev_2_summary.iloc[0]


checks = {
    "applprev_2成功读取": (
        int(
            summary_row[
                "child_row_count"
            ]
        )
        > 0
    ),

    "存在可聚合特征字段": (
        len(applprev_2_feature_schema)
        > 0
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 5.5检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


print(
    "Step 5.5检查："
    f"{len(checks)}/{len(checks)}通过"
)

print(
    "读取的applprev_2文件：",
    [
        file_path.name
        for file_path
        in applprev_2_files
    ]
)

print(
    "可聚合字段数：",
    len(applprev_2_feature_schema)
)

display(applprev_2_feature_schema)
display(applprev_2_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 5.5检查：2/2通过
读取的applprev_2文件： ['train_applprev_2.parquet']
可聚合字段数： 3


,column_name,column_type
0,cacccardblochreas_147M,VARCHAR
1,conts_type_509L,VARCHAR
2,credacc_cards_status_52L,VARCHAR


,child_row_count,parent_application_count,case_count_with_child_history,average_children_per_parent,maximum_children_per_parent
0,12371361.0,5707370,1101236,2.17,12


In [15]:
# Step 5.6：applprev_2两级聚合

connection.execute(
    """
    DROP TABLE IF EXISTS
        applprev_2_features_dev
    """
)


applprev_2_categorical_columns = (
    applprev_2_feature_schema[
        "column_name"
    ].tolist()
)


# ==========================================
# 第一层：子记录 → 一笔历史申请
# ==========================================

parent_level_expressions = [
    """
    COUNT(*) AS
        child_record_count
    """
]


# ==========================================
# 第二层：历史申请 → 当前case_id
# ==========================================

case_level_expressions = [
    """
    COUNT(*) AS
        applprev2__parent_application_count
    """,

    """
    SUM(child_record_count) AS
        applprev2__record_count
    """,

    """
    CAST(
        AVG(child_record_count)
        AS FLOAT
    ) AS
        applprev2__children_per_parent_mean
    """,

    """
    MIN(child_record_count) AS
        applprev2__children_per_parent_min
    """,

    """
    MAX(child_record_count) AS
        applprev2__children_per_parent_max
    """
]


for column_name in (
    applprev_2_categorical_columns
):

    column_sql = quote_identifier(
        column_name
    )

    prefix = (
        f"applprev2__{column_name}"
    )

    non_null_count_alias = (
        f"{prefix}__non_null_count"
    )

    parent_value_alias = (
        f"{prefix}__lowest_group2_value"
    )


    # 第一层：在每笔历史申请内部聚合

    parent_level_expressions.extend(
        [
            f"""
            COUNT(
                {column_sql}
            ) AS
                "{non_null_count_alias}"
            """,

            f"""
            ARG_MIN(
                {column_sql},
                num_group2
            ) AS
                "{parent_value_alias}"
            """
        ]
    )


    # 第二层：在当前case_id层面再次聚合

    case_level_expressions.extend(
        [
            f"""
            CAST(
                SUM(
                    "{non_null_count_alias}"
                )
                * 1.0
                /
                NULLIF(
                    SUM(child_record_count),
                    0
                )
                AS FLOAT
            ) AS
                "{prefix}__non_null_rate"
            """,

            f"""
            ARG_MIN(
                "{parent_value_alias}",
                num_group1
            ) AS
                "{prefix}__lowest_hierarchy_value"
            """
        ]
    )


parent_level_sql = (
    ",\n".join(
        parent_level_expressions
    )
)

case_level_sql = (
    ",\n".join(
        case_level_expressions
    )
)


connection.execute(
    f"""
    CREATE OR REPLACE TEMPORARY TABLE
        applprev_2_features_dev AS

    WITH child_by_parent AS (
        SELECT
            case_id,
            num_group1,

            {parent_level_sql}

        FROM applprev_2_dev_raw

        GROUP BY
            case_id,
            num_group1
    )

    SELECT
        case_id,

        {case_level_sql}

    FROM child_by_parent

    GROUP BY
        case_id
    """
)


print(
    "applprev_2两级聚合完成。"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

applprev_2两级聚合完成。


In [16]:
# Step 5.6：聚合结果检查

applprev_2_aggregation_check = (
    connection.execute(
        """
        SELECT
            COUNT(*) AS feature_row_count,

            COUNT(DISTINCT case_id)
                AS unique_case_id_count,

            SUM(
                applprev2__parent_application_count
            ) AS aggregated_parent_count,

            SUM(
                applprev2__record_count
            ) AS aggregated_child_record_count,

            MIN(
                applprev2__children_per_parent_min
            ) AS minimum_children_per_parent,

            MAX(
                applprev2__children_per_parent_max
            ) AS maximum_children_per_parent

        FROM applprev_2_features_dev
        """
    ).df()
)


applprev_2_column_count = len(
    connection.execute(
        """
        DESCRIBE applprev_2_features_dev
        """
    ).fetchall()
)


applprev_2_check_row = (
    applprev_2_aggregation_check.iloc[0]
)


expected_applprev_2_column_count = (
    1                       # case_id
    + 5                     # 基础数量特征
    + 2 * len(
        applprev_2_categorical_columns
    )
)


checks = {
    "聚合后一行一个case_id": (
        int(
            applprev_2_check_row[
                "feature_row_count"
            ]
        )
        ==
        int(
            applprev_2_check_row[
                "unique_case_id_count"
            ]
        )
    ),

    "案件覆盖数量正确": (
        int(
            applprev_2_check_row[
                "feature_row_count"
            ]
        )
        == 1_101_236
    ),

    "父历史申请数量没有丢失": (
        int(
            applprev_2_check_row[
                "aggregated_parent_count"
            ]
        )
        == 5_707_370
    ),

    "Depth 2子记录没有丢失": (
        int(
            applprev_2_check_row[
                "aggregated_child_record_count"
            ]
        )
        == 12_371_361
    ),

    "聚合字段数正确": (
        applprev_2_column_count
        == expected_applprev_2_column_count
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 5.6检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


print(
    "Step 5.6检查："
    f"{len(checks)}/{len(checks)}通过"
)

print(
    "聚合后数据形状：",
    (
        int(
            applprev_2_check_row[
                "feature_row_count"
            ]
        ),
        applprev_2_column_count
    )
)

display(
    applprev_2_aggregation_check
)

Step 5.6检查：5/5通过
聚合后数据形状： (1101236, 12)


,feature_row_count,unique_case_id_count,aggregated_parent_count,aggregated_child_record_count,minimum_children_per_parent,maximum_children_per_parent
0,1101236,1101236,5707370.0,12371361.0,1,12


In [17]:
# Step 5.7.1：导出applprev_2聚合特征

APPLPREV2_FEATURE_PATH = (
    DEPTH1_FEATURE_DIR
    / "applprev_2_features_dev_v1.parquet"
)


if APPLPREV2_FEATURE_PATH.exists():

    print(
        "applprev_2特征文件已经存在，"
        "跳过重复导出。"
    )

else:

    connection.execute(
        f"""
        COPY (
            SELECT *
            FROM applprev_2_features_dev
        )

        TO {sql_text(APPLPREV2_FEATURE_PATH)}

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE 100000
        )
        """
    )

    print(
        "applprev_2历史特征导出成功。"
    )


print(
    "输出文件：",
    APPLPREV2_FEATURE_PATH
)

applprev_2特征文件已经存在，跳过重复导出。
输出文件： D:\Risk_control project\data\processed\depth1\applprev_2_features_dev_v1.parquet


In [18]:
# Step 5.7.2：接入applprev_2特征

connection.execute(
    f"""
    CREATE OR REPLACE VIEW
        feature_mart_applprev_dev AS

    SELECT
        mart.*,

        CAST(
            COALESCE(
                depth2.applprev2__parent_application_count,
                0
            )
            AS INTEGER
        ) AS applprev2__parent_application_count,

        CAST(
            COALESCE(
                depth2.applprev2__record_count,
                0
            )
            AS INTEGER
        ) AS applprev2__record_count,

        depth2.* EXCLUDE (
            case_id,
            applprev2__parent_application_count,
            applprev2__record_count
        )

    FROM feature_mart_applprev1_dev
        AS mart

    LEFT JOIN read_parquet(
        {sql_text(APPLPREV2_FEATURE_PATH)}
    ) AS depth2

        ON mart.case_id
            = depth2.case_id
    """
)


print(
    "applprev_1与applprev_2"
    "工作Feature Mart创建成功。"
)

applprev_1与applprev_2工作Feature Mart创建成功。


In [19]:
# Step 5.7.3：导出与连接检查

applprev_integration_check = (
    connection.execute(
        """
        SELECT
            COUNT(*) AS row_count,

            COUNT(DISTINCT case_id)
                AS unique_case_id_count,

            SUM(
                applprev2__parent_application_count
            ) AS total_parent_application_count,

            SUM(
                applprev2__record_count
            ) AS total_depth2_record_count

        FROM feature_mart_applprev_dev
        """
    ).df()
)


applprev2_export_column_count = len(
    connection.execute(
        f"""
        DESCRIBE
        SELECT *

        FROM read_parquet(
            {sql_text(APPLPREV2_FEATURE_PATH)}
        )
        """
    ).fetchall()
)


applprev_joined_column_count = len(
    connection.execute(
        """
        DESCRIBE feature_mart_applprev_dev
        """
    ).fetchall()
)


check_row = (
    applprev_integration_check.iloc[0]
)

expected_applprev_joined_column_count = (
    joined_column_count
    + applprev2_export_column_count
    - 1
)

checks = {
    "applprev_2导出字段数正确": (
        applprev2_export_column_count
        == 12
    ),

    "连接没有改变样本行数": (
        int(check_row["row_count"])
        == 1_401_854
    ),

    "连接后仍一行一个case_id": (
        int(
            check_row[
                "unique_case_id_count"
            ]
        )
        == 1_401_854
    ),

    "父历史申请数量正确": (
        int(
            check_row[
                "total_parent_application_count"
            ]
        )
        == 5_707_370
    ),

    "Depth 2记录数量正确": (
        int(
            check_row[
                "total_depth2_record_count"
            ]
        )
        == 12_371_361
    ),

    "连接后字段数正确": (
        applprev_joined_column_count
        == expected_applprev_joined_column_count
    )
}


failed_checks = [
    check_name
    for check_name, passed
    in checks.items()
    if not passed
]


if failed_checks:
    raise ValueError(
        "Step 5.7检查失败：\n- "
        + "\n- ".join(failed_checks)
    )


file_size_mb = (
    APPLPREV2_FEATURE_PATH.stat().st_size
    / 1024**2
)


print(
    "Step 5.7检查："
    f"{len(checks)}/{len(checks)}通过"
)

print(
    "applprev_2文件大小：",
    f"{file_size_mb:.2f} MB"
)

print(
    "当前工作Feature Mart形状：",
    (
        int(check_row["row_count"]),
        applprev_joined_column_count
    )
)

display(
    applprev_integration_check
)

Step 5.7检查：6/6通过
applprev_2文件大小： 5.27 MB
当前工作Feature Mart形状： (1401854, 338)


,row_count,unique_case_id_count,total_parent_application_count,total_depth2_record_count
0,1401854,1401854,5707370.0,12371361.0


# Step 5.8：建立通用Depth 1聚合函数
## 定位Parquet
→ 限定开发期case_id
→ 自动识别字段类型
→ 聚合为一行一个case_id
→ 直接导出Parquet
→ 简短检查

In [26]:
# 修正版Step 5.8.1：通用Depth 1聚合函数

DEPTH1_NUMERIC_TYPE_PREFIXES = (
    "TINYINT",
    "SMALLINT",
    "INTEGER",
    "BIGINT",
    "HUGEINT",
    "UTINYINT",
    "USMALLINT",
    "UINTEGER",
    "UBIGINT",
    "FLOAT",
    "DOUBLE",
    "REAL",
    "DECIMAL"
)


def build_depth1_features(
    table_id: str,
    file_pattern: str,
    feature_prefix: str,
    pit_anchor_column="openingdate_857D",
    overwrite=True
):
    """
    将一张Depth 1历史表聚合为一行一个case_id，
    并直接导出为Parquet。

    参数
    ----------
    table_id:
        逻辑表名称，例如debitcard_1。

    file_pattern:
        物理文件匹配表达式，例如
        train_debitcard_1*.parquet。

    feature_prefix:
        输出特征前缀，例如debitcard1。

    pit_anchor_column:
        可选的记录产生日期。
        只有明确属于记录产生时间时才填写。

    overwrite:
        是否删除并重建已经存在的派生文件。
    """

    # ======================================
    # 1. 定位物理文件
    # ======================================

    table_files = sorted(
        TRAIN_DATA_DIR.glob(
            file_pattern
        )
    )


    if not table_files:

        alternative_train_dir = (
            DATA_ROOT
            / "train"
        )

        table_files = sorted(
            alternative_train_dir.glob(
                file_pattern
            )
        )


    if not table_files:
        raise FileNotFoundError(
            f"找不到文件：{file_pattern}"
        )


    table_files_sql = (
        "["
        + ", ".join(
            sql_text(file_path)
            for file_path in table_files
        )
        + "]"
    )


    raw_view_name = (
        f"{table_id}_dev_raw"
    )

    eligible_view_name = (
        f"{table_id}_dev_eligible"
    )


    # ======================================
    # 2. 限定开发期案件
    # ======================================

    connection.execute(
        f"""
        CREATE OR REPLACE VIEW
            {raw_view_name} AS

        SELECT
            history.*,

            base.date_decision
                AS current_date_decision

        FROM read_parquet(
            {table_files_sql},
            union_by_name = true
        ) AS history

        INNER JOIN base_dev_keys AS base
            ON history.case_id
                = base.case_id
        """
    )


    raw_schema = connection.execute(
        f"""
        DESCRIBE {raw_view_name}
        """
    ).df()


    raw_column_names = set(
        raw_schema["column_name"]
    )


    required_columns = {
        "case_id",
        "num_group1"
    }


    missing_required_columns = (
        required_columns
        - raw_column_names
    )


    if missing_required_columns:
        raise ValueError(
            f"{table_id}缺少Depth 1必要字段："
            + str(
                sorted(
                    missing_required_columns
                )
            )
        )


    # ======================================
    # 3. 可选的时点过滤
    # ======================================

    excluded_future_record_count = 0


    if pit_anchor_column is not None:

        if (
            pit_anchor_column
            not in raw_column_names
        ):
            raise ValueError(
                f"{table_id}中不存在"
                f"{pit_anchor_column}"
            )


        pit_anchor_sql = quote_identifier(
            pit_anchor_column
        )


        # 只有明确作为时点过滤锚点的字段，
        # 才要求全部非空值能够转换为日期。

        pit_anchor_parse_check = (
            connection.execute(
                f"""
                SELECT
                    SUM(
                        CASE
                            WHEN
                                {pit_anchor_sql}
                                IS NOT NULL
                            THEN 1
                            ELSE 0
                        END
                    ) AS non_null_count,

                    SUM(
                        CASE
                            WHEN
                                {pit_anchor_sql}
                                IS NOT NULL

                                AND

                                TRY_CAST(
                                    {pit_anchor_sql}
                                    AS DATE
                                ) IS NULL
                            THEN 1
                            ELSE 0
                        END
                    ) AS parse_failure_count

                FROM {raw_view_name}
                """
            ).df()
        )


        parse_failure_count = int(
            pit_anchor_parse_check.loc[
                0,
                "parse_failure_count"
            ]
            or 0
        )


        if parse_failure_count > 0:
            raise ValueError(
                f"{table_id}的时点锚点"
                f"{pit_anchor_column}"
                f"存在{parse_failure_count:,}"
                "个无法转换为日期的非空值。"
            )


        excluded_future_record_count = (
            connection.execute(
                f"""
                SELECT COUNT(*)

                FROM {raw_view_name}

                WHERE
                    TRY_CAST(
                        {pit_anchor_sql}
                        AS DATE
                    )
                    >
                    CAST(
                        current_date_decision
                        AS DATE
                    )
                """
            ).fetchone()[0]
        )


        connection.execute(
            f"""
            CREATE OR REPLACE VIEW
                {eligible_view_name} AS

            SELECT
                *

            FROM {raw_view_name}

            WHERE
                {pit_anchor_sql} IS NULL

                OR

                TRY_CAST(
                    {pit_anchor_sql}
                    AS DATE
                )
                <=
                CAST(
                    current_date_decision
                    AS DATE
                )
            """
        )


    else:

        connection.execute(
            f"""
            CREATE OR REPLACE VIEW
                {eligible_view_name} AS

            SELECT
                *

            FROM {raw_view_name}
            """
        )



    # ======================================
    # 过滤后记录数检查
    # ======================================


    source_row_count = (
        connection.execute(
            f"""
            SELECT COUNT(*)

            FROM {eligible_view_name}
            """
        ).fetchone()[0]
    )


    if source_row_count == 0:
        raise ValueError(
            f"{table_id}在开发期和时点过滤后为0行，"
            "因此不生成特征文件。"
        )

    # ======================================
    # 4. 自动识别字段类型
    # ======================================

    eligible_schema = connection.execute(
        f"""
        DESCRIBE {eligible_view_name}
        """
    ).df()


    technical_columns = {
        "case_id",
        "num_group1",
        "num_group2",
        "filename",
        "current_date_decision",
        "current_week_num",
        "date_decision",
        "WEEK_NUM",
        "target"
    }


    numeric_columns = []
    date_columns = []
    boolean_columns = []
    categorical_columns = []


    for _, schema_row in (
        eligible_schema.iterrows()
    ):

        column_name = (
            schema_row["column_name"]
        )

        column_type = str(
            schema_row["column_type"]
        ).upper()


        if column_name in technical_columns:
            continue


        # Home Credit字段后缀D表示日期。
        # 部分日期在Parquet中可能被读取为VARCHAR，
        # 因此字段名称规则优先于物理数据类型。

        if (
            column_name.upper().endswith(
                "D"
            )
            or
            column_type.startswith("DATE")
            or
            column_type.startswith(
                "TIMESTAMP"
            )
        ):
            date_columns.append(
                column_name
            )


        elif column_type.startswith(
            DEPTH1_NUMERIC_TYPE_PREFIXES
        ):
            numeric_columns.append(
                column_name
            )


        elif column_type.startswith(
            "BOOLEAN"
        ):
            boolean_columns.append(
                column_name
            )


        else:
            categorical_columns.append(
                column_name
            )


    # ======================================
    # 5. 构造聚合表达式
    # ======================================

    aggregation_expressions = [
        f"""
        COUNT(*) AS
            {feature_prefix}__record_count
        """
    ]


    # --------------------------------------
    # 数值字段
    # --------------------------------------

    for column_name in numeric_columns:

        column_sql = quote_identifier(
            column_name
        )

        prefix = (
            f"{feature_prefix}"
            f"__{column_name}"
        )


        aggregation_expressions.extend(
            [
                f"""
                CAST(
                    AVG(
                        CAST(
                            {column_sql}
                            AS DOUBLE
                        )
                    )
                    AS FLOAT
                ) AS "{prefix}__mean"
                """,

                f"""
                CAST(
                    MIN(
                        CAST(
                            {column_sql}
                            AS DOUBLE
                        )
                    )
                    AS FLOAT
                ) AS "{prefix}__min"
                """,

                f"""
                CAST(
                    MAX(
                        CAST(
                            {column_sql}
                            AS DOUBLE
                        )
                    )
                    AS FLOAT
                ) AS "{prefix}__max"
                """
            ]
        )


        # Home Credit后缀A通常表示金额。
        # 金额字段额外计算历史合计值。

        if column_name.upper().endswith(
            "A"
        ):
            aggregation_expressions.append(
                f"""
                CAST(
                    SUM(
                        CAST(
                            {column_sql}
                            AS DOUBLE
                        )
                    )
                    AS FLOAT
                ) AS "{prefix}__sum"
                """
            )


    # --------------------------------------
    # 日期字段
    # --------------------------------------

    for column_name in date_columns:

        column_sql = quote_identifier(
            column_name
        )

        prefix = (
            f"{feature_prefix}"
            f"__{column_name}"
        )


        days_expression = f"""
            DATE_DIFF(
                'day',

                TRY_CAST(
                    {column_sql}
                    AS DATE
                ),

                CAST(
                    current_date_decision
                    AS DATE
                )
            )
        """


        aggregation_expressions.extend(
            [
                f"""
                MIN(
                    {days_expression}
                ) AS
                    "{prefix}__days_to_decision_min"
                """,

                f"""
                MAX(
                    {days_expression}
                ) AS
                    "{prefix}__days_to_decision_max"
                """,

                f"""
                CAST(
                    AVG(
                        {days_expression}
                    )
                    AS FLOAT
                ) AS
                    "{prefix}__days_to_decision_mean"
                """
            ]
        )


    # --------------------------------------
    # 布尔字段
    # --------------------------------------

    for column_name in boolean_columns:

        column_sql = quote_identifier(
            column_name
        )

        prefix = (
            f"{feature_prefix}"
            f"__{column_name}"
        )


        aggregation_expressions.append(
            f"""
            CAST(
                AVG(
                    CASE
                        WHEN
                            {column_sql}
                            = TRUE
                        THEN 1.0

                        WHEN
                            {column_sql}
                            = FALSE
                        THEN 0.0

                        ELSE NULL
                    END
                )
                AS FLOAT
            ) AS
                "{prefix}__true_rate"
            """
        )


    # --------------------------------------
    # 类别字段
    # --------------------------------------

    for column_name in (
        categorical_columns
    ):

        column_sql = quote_identifier(
            column_name
        )

        prefix = (
            f"{feature_prefix}"
            f"__{column_name}"
        )


        aggregation_expressions.append(
            f"""
            ARG_MIN(
                {column_sql},
                num_group1
            ) AS
                "{prefix}__lowest_group_index_value"
            """
        )


    aggregation_sql = (
        ",\n".join(
            aggregation_expressions
        )
    )


    # ======================================
    # 6. 设置输出路径
    # ======================================

    output_path = (
        DEPTH1_FEATURE_DIR
        / (
            f"{table_id}"
            "_features_dev_v1.parquet"
        )
    )


    # overwrite=True仅删除这张可重新生成的
    # 派生特征文件，不影响任何原始数据。

    if overwrite and output_path.exists():

        output_path.unlink()

        print(
            f"{table_id}旧派生文件"
            "已删除并准备重建。"
        )


    # ======================================
    # 7. 执行聚合并导出
    # ======================================

    if output_path.exists():

        print(
            f"{table_id}特征文件已存在，"
            "跳过重复导出。"
        )

    else:

        connection.execute(
            f"""
            COPY (
                SELECT
                    case_id,

                    {aggregation_sql}

                FROM {eligible_view_name}

                GROUP BY
                    case_id
            )

            TO {sql_text(output_path)}

            (
                FORMAT PARQUET,
                COMPRESSION ZSTD,
                ROW_GROUP_SIZE 100000
            )
            """
        )

        print(
            f"{table_id}特征文件"
            "生成成功。"
        )


    # ======================================
    # 8. 简短结果检查
    # ======================================

    source_row_count = (
        connection.execute(
            f"""
            SELECT COUNT(*)

            FROM {eligible_view_name}
            """
        ).fetchone()[0]
    )


    output_summary = connection.execute(
        f"""
        SELECT
            COUNT(*) AS feature_row_count,

            COUNT(DISTINCT case_id)
                AS unique_case_id_count,

            COALESCE(
                SUM(
                    "{feature_prefix}__record_count"
                ),
                0
            ) AS aggregated_source_row_count

        FROM read_parquet(
            {sql_text(output_path)}
        )
        """
    ).df()


    output_column_count = len(
        connection.execute(
            f"""
            DESCRIBE
            SELECT *

            FROM read_parquet(
                {sql_text(output_path)}
            )
            """
        ).fetchall()
    )


    summary_row = (
        output_summary.iloc[0]
    )


    checks = {
        "输出一行一个case_id": (
            int(
                summary_row[
                    "feature_row_count"
                ]
            )
            ==
            int(
                summary_row[
                    "unique_case_id_count"
                ]
            )
        ),

        "聚合记录数正确": (
            int(
                summary_row[
                    "aggregated_source_row_count"
                ]
            )
            ==
            int(source_row_count)
        ),

        "输出文件创建成功": (
            output_path.exists()
        )
    }


    failed_checks = [
        check_name
        for check_name, passed
        in checks.items()
        if not passed
    ]


    if failed_checks:
        raise ValueError(
            f"{table_id}检查失败：\n- "
            + "\n- ".join(
                failed_checks
            )
        )


    file_size_mb = (
        output_path.stat().st_size
        / 1024**2
    )


    print(
        f"{table_id}检查："
        f"{len(checks)}/"
        f"{len(checks)}通过"
    )

    print(
        "源记录数：",
        f"{source_row_count:,}"
    )

    print(
        "聚合后案件数：",
        f"{int(summary_row['feature_row_count']):,}"
    )

    print(
        "输出字段数：",
        output_column_count
    )

    print(
        "文件大小：",
        f"{file_size_mb:.2f} MB"
    )


    if pit_anchor_column is not None:

        print(
            "时点过滤排除记录数：",
            f"{excluded_future_record_count:,}"
        )


    return {
        "table_id":
            table_id,

        "output_path":
            output_path,

        "summary":
            output_summary,

        "column_count":
            output_column_count,

        "numeric_columns":
            numeric_columns,

        "date_columns":
            date_columns,

        "boolean_columns":
            boolean_columns,

        "categorical_columns":
            categorical_columns,

        "excluded_future_record_count":
            excluded_future_record_count
    }

In [27]:
# 修正版Step 5.8.2：重新生成debitcard_1

debitcard_1_result = (
    build_depth1_features(
        table_id="debitcard_1",

        file_pattern=(
            "train_debitcard_1"
            "*.parquet"
        ),

        feature_prefix="debitcard1",

        pit_anchor_column="openingdate_857D",

        overwrite=True
    )
)


display(
    debitcard_1_result[
        "summary"
    ]
)


print(
    "数值字段：",
    debitcard_1_result[
        "numeric_columns"
    ]
)

print(
    "日期字段：",
    debitcard_1_result[
        "date_columns"
    ]
)

print(
    "布尔字段：",
    debitcard_1_result[
        "boolean_columns"
    ]
)

print(
    "类别字段：",
    debitcard_1_result[
        "categorical_columns"
    ]
)

debitcard_1旧派生文件已删除并准备重建。
debitcard_1特征文件生成成功。
debitcard_1检查：3/3通过
源记录数： 142,446
聚合后案件数： 101,108
输出字段数： 17
文件大小： 0.94 MB
时点过滤排除记录数： 0


,feature_row_count,unique_case_id_count,aggregated_source_row_count
0,101108,101108,142446.0


数值字段： ['last180dayaveragebalance_704A', 'last180dayturnover_1134A', 'last30dayturnover_651A']
日期字段： ['openingdate_857D']
布尔字段： []
类别字段： []


In [25]:
# Step 5.9：批量处理简单Depth 1表

simple_depth1_configs = [
    {
        "table_id":
            "deposit_1",

        "file_pattern":
            "train_deposit_1*.parquet",

        "feature_prefix":
            "deposit1",

        "pit_anchor_column":
            "openingdate_313D"
    },

    {
        "table_id":
            "other_1",

        "file_pattern":
            "train_other_1*.parquet",

        "feature_prefix":
            "other1",

        "pit_anchor_column":
            None
    },

    {
        "table_id":
            "tax_registry_b_1",

        "file_pattern":
            "train_tax_registry_b_1*.parquet",

        "feature_prefix":
            "taxregistryb1",

        "pit_anchor_column":
            "deductiondate_4917603D"
    },

    {
        "table_id":
            "tax_registry_c_1",

        "file_pattern":
            "train_tax_registry_c_1*.parquet",

        "feature_prefix":
            "taxregistryc1",

        "pit_anchor_column":
            "processingdate_168D"
    }
]


simple_depth1_results = {}


for config in simple_depth1_configs:

    print(
        "\n正在处理：",
        config["table_id"]
    )

    result = build_depth1_features(
        table_id=(
            config["table_id"]
        ),

        file_pattern=(
            config["file_pattern"]
        ),

        feature_prefix=(
            config["feature_prefix"]
        ),

        pit_anchor_column=(
            config["pit_anchor_column"]
        ),

        overwrite=False
    )

    simple_depth1_results[
        config["table_id"]
    ] = result


正在处理： deposit_1
deposit_1特征文件已存在，跳过重复导出。
deposit_1检查：3/3通过
源记录数： 131,383
聚合后案件数： 95,065
输出字段数： 12
文件大小： 1.83 MB
时点过滤排除记录数： 0

正在处理： other_1
other_1特征文件已存在，跳过重复导出。
other_1检查：3/3通过
源记录数： 35,004
聚合后案件数： 35,004
输出字段数： 22
文件大小： 1.03 MB

正在处理： tax_registry_b_1
tax_registry_b_1特征文件生成成功。
tax_registry_b_1检查：3/3通过
源记录数： 487,863
聚合后案件数： 69,268
输出字段数： 10
文件大小： 1.39 MB
时点过滤排除记录数： 40,021

正在处理： tax_registry_c_1
tax_registry_c_1特征文件生成成功。
tax_registry_c_1检查：3/3通过
源记录数： 2,617,445
聚合后案件数： 403,018
输出字段数： 10
文件大小： 8.09 MB
时点过滤排除记录数： 726,355
